In [ ]:
import plotly.express as px
def plot_kNN_alignment_scores(fig):
    fig.update_layout(
            yaxis_title="Mean KNN feature<br>alignment score",
            template="plotly_white",
            font={"family": "Arial", "csolor": "black", "size": 12},
            legend_title_text="Embedding models",
            width=1200,
            height=500,
    )

    fig.update_traces(marker=dict(size=10), line=dict(width=4))

    for axis in fig.layout:
        if axis.startswith('xaxis'):
            fig.layout[axis].update(
                showgrid=False,
                linecolor='black',
                linewidth=3,
                ticks='outside',
                tickwidth=2,
                tickcolor='black',
                ticklen=6,
                tickformat=".1f",
                dtick=0.1
            )
        if axis.startswith('yaxis'):
            fig.layout[axis].update(
                showgrid=False,
                linecolor='black',
                linewidth=3,
                ticks='outside',
                tickwidth=2,
                tickcolor='black',
                ticklen=6,
                tickformat=".2f",
                dtick=0.05
            )    

    for annotation in fig.layout.annotations:
        if '=' in annotation.text:
            annotation.text = annotation.text.split('=')[1]
                
    fig.update_layout(
        title=dict(
            x=0,
            xanchor='left'
        ),
        legend=dict(
            orientation='h',
            x=0.48,
            y=1.05,
            xanchor='center',
            yanchor='bottom'
        ),
        margin=dict(t=120)
    )
    return fig


In [18]:
import pickle
import sys
import pandas as pd
import plotly.colors as pc
import plotly.express as px
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from constants import IMG_OUTPUT_PATH, EMB_SPACES

# MLP results
mlp_performance_results = {}
for emb_space in EMB_SPACES:
    with open(f'{IMG_OUTPUT_PATH}/performance/train2-epochs=8/{emb_space[0]}_torch.pkl', 'rb') as f:
        mlp_performance_results[emb_space[0]] = pickle.load(f)

# kNN results
collected_results = []
for random_shuffle in [0, 0.1, 0.2, 0.5, 1.0]:
    for metric in ['cityblock','cosine', 'euclidean']:
        with open(f'{IMG_OUTPUT_PATH}/shuffle-dataset/{metric}/{random_shuffle}.pkl', 'rb') as f:
            performance_results = pickle.load(f)
            for emb_space in EMB_SPACES:
                emb_space_name = emb_space[0]
                kNN_score = performance_results[emb_space_name].mean()
                performance = float(mlp_performance_results[emb_space_name]['test_auprc'])
                legend_text = f'{performance:.3f} ({emb_space_name})'
                metric_description = 'Manhattan distance' if metric == 'cityblock' else f'{metric.capitalize()} distance'
                collected_results.append((random_shuffle, emb_space_name, kNN_score, metric_description , legend_text, performance))

df = pd.DataFrame(collected_results, columns=['shuffle_ratio', 'Embedding Space', 'kNN score', 'metric', 'Legend text', 'Performance'])


sorted_performance = df[['Performance', 'Legend text', 'Embedding Space']].drop_duplicates().sort_values('Performance', ascending=True)
sorted_performance_descending = df[['Performance', 'Legend text', 'Embedding Space']].drop_duplicates().sort_values('Performance', ascending=False)
n_bins = len(sorted_performance)
low, high = 0.3, 1.0
color_scale = pc.sample_colorscale(
        "Reds",
        [low + (high - low) * i / (n_bins - 1) for i in range(n_bins)],
        colortype='rgb'
        )
label_to_color ={}
for label, color in zip(sorted_performance['Legend text'].tolist(), color_scale):
    label_to_color[label] = color

# plot
fig = px.line(
    df,
    x="shuffle_ratio",
    y="kNN score",
    color="Legend text",
    color_discrete_map=label_to_color,
    facet_col="metric",
    markers=True,
    category_orders={"Legend text": sorted_performance_descending['Legend text'].tolist()}
)

y=0.33333

fig.add_hline(y=y, line_dash="dot",
              fillcolor='grey',
)
fig = plot_kNN_alignment_scores(fig)
fig.show()